In [1]:
import os
import re
import time
import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import cosine_similarity

# New imports for the Deep Learning upgrade
import torch
from transformers import AutoTokenizer, AutoModel

class NBASemanticVectorPipeline:
    def __init__(self, target_players_df_path, output_csv_name="nlp_model_scores.csv", credits_csv_name="articles_credited.csv"):
        self.output_csv = output_csv_name
        self.credits_csv = credits_csv_name
        self.api_url = "https://en.wikipedia.org/w/api.php"
        self.headers = {"User-Agent": "NBASemanticBot/2.0 (sportsanalyticsproject@example.com)"}
        
        if os.path.exists(target_players_df_path):
            master_df = pd.read_csv(target_players_df_path)
            self.target_players = master_df['Player_Name'].dropna().unique().tolist()
            print(f"🎯 Master roster loaded. Tracking {len(self.target_players)} entries.")
        else:
            raise FileNotFoundError(f"Could not find your master CSV file at: {target_players_df_path}")

        # Core semantic anchor phrases representing your exact model traits
        self.concepts = {
            "Respect_Score": [
                "greatest basketball player of all time legend revered goat prestige iconic legacy",
                "dominant elite status hall of fame accolades unmatched peak supreme fear praise"
            ],
            "Failings_Score": [
                "career failings struggles collapse choked flaw weakness mistake breakdown disappointment",
                "criticized for missing clutch shots turnovers liability losing series internal downfall"
            ],
            "Positive_Attitude_Score": [
                "positive attitude leadership inspiring locker room presence unselfish teammate mentor",
                "charity outreach dedicated work ethic professional supportive captain champion chemistry"
            ],
            "Negative_Attitude_Score": [
                "negative attitude toxic arrogant selfish complaining frustrated feud distraction drama",
                "demanded trade clashed with coach team distraction arguments tension technical fouls"
            ]
        }

        # Initialize RoBERTa components
        print("🤗 Loading RoBERTa model and tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained("roberta-base")
        self.model = AutoModel.from_pretrained("roberta-base")
        
        # Performance Optimization: Pre-compute concept embeddings once at startup
        print("🧠 Pre-computing semantic anchor vectors...")
        self.concept_vectors = {}
        for score_name, anchor_phrases in self.concepts.items():
            vectors = [self._get_roberta_embedding(phrase) for phrase in anchor_phrases]
            # Average the vectors for the anchor phrases to get a single concept representation
            self.concept_vectors[score_name] = np.mean(vectors, axis=0)

    def _get_roberta_embedding(self, text):
        """Generates a dense semantic vector from RoBERTa using mean pooling."""
        # RoBERTa has a 512 token limit; truncation=True prevents crashes on massive text
        inputs = self.tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors="pt")
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            
        # Perform mean pooling to get a single vector representing the entire text chunk
        token_embeddings = outputs.last_hidden_state
        attention_mask = inputs['attention_mask']
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        
        return (sum_embeddings / sum_mask).numpy()

    def fetch_wikipedia_text_and_url(self, player_name):
        """Searches Wikipedia to get corrected page titles and extracts full career biographies."""
        search_params = {
            "action": "query", "list": "search", "srsearch": player_name, "format": "json", "srlimit": 1
        }
        
        try:
            search_res = requests.get(self.api_url, params=search_params, headers=self.headers, timeout=10)
            if search_res.status_code == 200:
                search_data = search_res.json()
                search_results = search_data.get("query", {}).get("search", [])
                if not search_results:
                    return "", ""
                corrected_title = search_results[0]["title"]
            else:
                return "", ""
                
            time.sleep(1)
            parse_params = {
                "action": "query", "format": "json", "titles": corrected_title,
                "prop": "extracts|info", "exintro": 0, "explaintext": 1, "inprop": "url"
            }
            
            response = requests.get(self.api_url, params=parse_params, headers=self.headers, timeout=10)
            if response.status_code == 200:
                pages = response.json().get("query", {}).get("pages", {})
                for page_id, page_data in pages.items():
                    if page_id != "-1":
                        return page_data.get("extract", ""), page_data.get("fullurl", "")
        except Exception as e:
            print(f"⚠️ Connection glitch during fetch for {player_name}: {e}")
            
        return "", ""

    def extract_player_context(self, player_name, text):
        """Extracts text segments centered around the player's name variants."""
        if not text or len(text.strip()) < 100:
            return ""

        sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', text.strip())
        relevant_segments = []
        
        name_parts = player_name.split()
        last_name = name_parts[-1] if len(name_parts) > 1 else ""
        first_name = name_parts[0]
        
        aliases = [player_name.lower()]
        if len(last_name) > 3: aliases.append(last_name.lower())
        if len(first_name) > 3: aliases.append(first_name.lower())

        for idx, sentence in enumerate(sentences):
            sentence_lower = sentence.lower()
            if any(alias in sentence_lower for alias in aliases):
                start = max(0, idx - 2)
                end = min(len(sentences), idx + 3)
                relevant_segments.append(" ".join(sentences[start:end]))

        return " ".join(relevant_segments)

    def calculate_semantic_similarity(self, context_text):
        """Maps content vectors into cosine mathematical angles using transformer embeddings."""
        scores = {}
        # Get the deep-learning vector representation of the player's context chunk
        context_vector = self._get_roberta_embedding(context_text)
        
        for score_name, concept_vector in self.concept_vectors.items():
            # Calculate similarity between context and the pre-computed concept anchor vector
            similarity = cosine_similarity(context_vector, concept_vector)
            
            # Scale from [0, 1] to [0, 100] for pipeline compatibility
            scores[score_name] = round(float(similarity[0][0]) * 100, 3)
        return scores

    def run_pipeline(self):
        all_results = []
        
        for player in self.target_players:
            wiki_text, wiki_url = self.fetch_wikipedia_text_and_url(player)
            context_chunk = self.extract_player_context(player, wiki_text)
            
            if context_chunk and len(context_chunk.split()) > 15:
                print(f"🧠 Extracting transformer vectors for: {player}...")
                scores = self.calculate_semantic_similarity(context_chunk)
                
                player_result = {
                    "Player_Name": player,
                    "Context_Length": len(context_chunk),
                    "Respect_Score": scores["Respect_Score"],
                    "Failings_Score": scores["Failings_Score"],
                    "Positive_Attitude_Score": scores["Positive_Attitude_Score"],
                    "Negative_Attitude_Score": scores["Negative_Attitude_Score"]
                }
                all_results.append(player_result)
                
                new_credit = pd.DataFrame([{"Player_Name": player, "Source_URL": wiki_url}])
                if os.path.exists(self.credits_csv):
                    pd.concat([pd.read_csv(self.credits_csv), new_credit]).drop_duplicates().to_csv(self.credits_csv, index=False)
                else:
                    new_credit.to_csv(self.credits_csv, index=False)
            else:
                print(f"❌ Skipping {player} (Insufficient media text or missing search result found)")
                
            time.sleep(1.5)

        if all_results:
            new_df = pd.DataFrame(all_results)
            if os.path.exists(self.output_csv):
                existing_df = pd.read_csv(self.output_csv)
                existing_df.set_index("Player_Name", inplace=True)
                new_df.set_index("Player_Name", inplace=True)
                final_df = new_df.combine_first(existing_df).reset_index()
            else:
                final_df = new_df
                
            final_df.to_csv(self.output_csv, index=False)
            print(f"\n💾 Pipeline run complete! Files generated:\n📊 Metrics: '{self.output_csv}'\n🔗 Sources: '{self.credits_csv}'")

if __name__ == "__main__":
    master_file = os.path.join('..', 'Data Collection', 'nba_top_100_master.csv')
    
    engine = NBASemanticVectorPipeline(
        target_players_df_path=master_file,
        output_csv_name="nlp_model_scores.csv",
        credits_csv_name="articles_credited.csv"
    )
    engine.run_pipeline()

C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🎯 Master roster loaded. Tracking 114 entries.
🤗 Loading RoBERTa model and tokenizer...


C:\Users\gauth\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gauth\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 19

🧠 Pre-computing semantic anchor vectors...
🧠 Extracting transformer vectors for: Michael Jordan...
🧠 Extracting transformer vectors for: LeBron James...
🧠 Extracting transformer vectors for: Kareem Abdul-Jabbar...
🧠 Extracting transformer vectors for: Bill Russell...
🧠 Extracting transformer vectors for: Magic Johnson...
🧠 Extracting transformer vectors for: Wilt Chamberlain...
🧠 Extracting transformer vectors for: Shaquille O'Neal...
🧠 Extracting transformer vectors for: Tim Duncan...
🧠 Extracting transformer vectors for: Larry Bird...
🧠 Extracting transformer vectors for: Kobe Bryant...
🧠 Extracting transformer vectors for: Hakeem Olajuwon...
🧠 Extracting transformer vectors for: Stephen Curry...
🧠 Extracting transformer vectors for: Oscar Robertson...
🧠 Extracting transformer vectors for: Kevin Durant...
🧠 Extracting transformer vectors for: Julius Erving...
🧠 Extracting transformer vectors for: Jerry West...
🧠 Extracting transformer vectors for: Karl Malone...
🧠 Extracting transfor

In [2]:
import pandas as pd
import os

def rank_player_nlp_scores(input_csv="nlp_model_scores.csv", output_csv="nlp_model_scores_ranked.csv"):
    if not os.path.exists(input_csv):
        raise FileNotFoundError(f"Could not locate '{input_csv}'. Ensure your semantic engine script has run successfully first.")
        
    # Read the data
    df = pd.read_csv(input_csv)
    
    # Defensive check: Drop any players the API failed to fetch text for to avoid math errors
    df = df.dropna(subset=['Respect_Score', 'Positive_Attitude_Score', 'Failings_Score', 'Negative_Attitude_Score'])
    
    # Apply the semantic algorithm formula
    df['Overall_NLP_Score'] = (
        (df['Respect_Score'] + df['Positive_Attitude_Score']) - 
        (df['Failings_Score'] + df['Negative_Attitude_Score'])
    )
    
    # Clean up decimal precision to 3 places (Crucial for dense transformer embeddings)
    df['Overall_NLP_Score'] = df['Overall_NLP_Score'].round(3)
    
    # Sort values descending by the net score
    df_ranked = df.sort_values(by='Overall_NLP_Score', ascending=False).reset_index(drop=True)
    
    # Generate the official narrative standing tier ranks
    df_ranked['NLP_Rank'] = df_ranked.index + 1
    
    # Reorder columns to place key insights up front (Kept Context_Length for transparency)
    ordered_cols = [
        'NLP_Rank', 'Player_Name', 'Overall_NLP_Score', 
        'Respect_Score', 'Failings_Score', 'Positive_Attitude_Score', 'Negative_Attitude_Score',
        'Context_Length'
    ]
    
    # Filter to only columns that actually exist to avoid KeyErrors
    final_cols = [col for col in ordered_cols if col in df_ranked.columns]
    final_df = df_ranked[final_cols]
    
    # Overwrite/save the file with rankings included
    final_df.to_csv(output_csv, index=False)
    
    print(f"📊 Calculations complete! Successfully ranked {len(final_df)} players.")
    print(f"💾 File updated and exported to: '{output_csv}'")
    print("\n👑 Top 5 Narrative Leaders Preview:")
    print(final_df.head(5).to_string(index=False))

if __name__ == "__main__":
    rank_player_nlp_scores()

📊 Calculations complete! Successfully ranked 114 players.
💾 File updated and exported to: 'nlp_model_scores_ranked.csv'

👑 Top 5 Narrative Leaders Preview:
 NLP_Rank     Player_Name  Overall_NLP_Score  Respect_Score  Failings_Score  Positive_Attitude_Score  Negative_Attitude_Score  Context_Length
        1 Spencer Haywood              3.900         94.440          91.727                   92.112                   90.925             423
        2     Paul Arizin              3.567         94.308          91.849                   92.049                   90.941             488
        3    Nikola Jokic              2.912         94.700          92.719                   92.809                   91.878             544
        4       Sam Jones              2.908         93.121          91.100                   91.136                   90.249             557
        5    LeBron James              2.746         87.541          85.519                   85.098                   84.374         